# EEG Dataset Analysis for Presentation

This notebook analyzes EDF files in `dataverse_files` and creates presentation-ready visuals:
- 3+ comparison graphs between Healthy and Schizophrenia groups
- Correlation matrix heatmap
- Additional useful plots for explaining the dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import mne
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 7)

DATA_DIR = './dataverse_files'
print('Using data folder:', os.path.abspath(DATA_DIR))

In [ ]:
def infer_label(filename: str):
    name = filename.lower().strip()
    if name.startswith(('h', 'h_control')):
        return 'Healthy'
    if name.startswith(('s', 'sz', 'sz_patient')):
        return 'Schizophrenia'
    return None

def extract_record_features(edf_path: str):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    data = raw.get_data()  # shape: channels x time
    ch_names = raw.ch_names
    sfreq = raw.info.get('sfreq', np.nan)
    duration_sec = raw.n_times / sfreq if sfreq and not np.isnan(sfreq) else np.nan

    row = {
        'filename': os.path.basename(edf_path),
        'n_channels': len(ch_names),
        'n_samples': raw.n_times,
        'sfreq': sfreq,
        'duration_sec': duration_sec,
    }

    for i, ch in enumerate(ch_names):
        x = data[i]
        row[f'{ch}_mean'] = np.mean(x)
        row[f'{ch}_std'] = np.std(x)
        row[f'{ch}_rms'] = np.sqrt(np.mean(x**2))
        row[f'{ch}_ptp'] = np.ptp(x)

    return row

rows = []
skipped = []

for file in sorted(os.listdir(DATA_DIR)):
    if not file.lower().endswith('.edf'):
        continue
    label = infer_label(file)
    if label is None:
        skipped.append(file)
        continue
    path = os.path.join(DATA_DIR, file)
    try:
        row = extract_record_features(path)
        row['label'] = label
        rows.append(row)
    except Exception as e:
        skipped.append(f'{file} -> {e}')

df = pd.DataFrame(rows)
print('Loaded records:', len(df))
print('Label counts:')
print(df['label'].value_counts())
print('\nSkipped files:', len(skipped))

In [ ]:
display(df.head(3))
print('Total columns/features:', df.shape[1])

## 1) Core Comparison Graphs (required)

In [ ]:
# Comparison graph 1: Class distribution
plt.figure(figsize=(8,5))
ax = sns.countplot(data=df, x='label', palette='Set2')
ax.set_title('Class Distribution (Healthy vs Schizophrenia)')
ax.set_xlabel('Class')
ax.set_ylabel('Number of EEG Recordings')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.show()

# pick a robust channel feature for comparisons
preferred_bases = ['Fp1', 'Fp2', 'Cz', 'Fz', 'Pz', 'C3', 'C4']
std_cols = [c for c in df.columns if c.endswith('_std')]
mean_cols = [c for c in df.columns if c.endswith('_mean')]

target_std = None
for base in preferred_bases:
    candidate = f'{base}_std'
    if candidate in std_cols:
        target_std = candidate
        break
if target_std is None and std_cols:
    target_std = std_cols[0]

target_mean = None
for base in preferred_bases:
    candidate = f'{base}_mean'
    if candidate in mean_cols:
        target_mean = candidate
        break
if target_mean is None and mean_cols:
    target_mean = mean_cols[0]

# Comparison graph 2: boxplot
if target_std:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x='label', y=target_std, palette='Set3')
    sns.stripplot(data=df, x='label', y=target_std, color='black', alpha=0.5)
    plt.title(f'Comparison of {target_std} by Class')
    plt.xlabel('Class')
    plt.ylabel('Signal Std')
    plt.show()

# Comparison graph 3: violinplot
if target_mean:
    plt.figure(figsize=(8,5))
    sns.violinplot(data=df, x='label', y=target_mean, palette='Pastel1', inner='quartile')
    plt.title(f'Comparison of {target_mean} by Class')
    plt.xlabel('Class')
    plt.ylabel('Signal Mean')
    plt.show()

## 2) Correlation Matrix (required)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Reduce to top-variance columns for readability
top_n = 25
var_series = numeric_df.var().sort_values(ascending=False)
top_cols = var_series.head(top_n).index.tolist()
corr = numeric_df[top_cols].corr()

plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, square=False)
plt.title(f'Correlation Matrix (Top {top_n} Numeric Features by Variance)')
plt.tight_layout()
plt.show()

## 3) Additional Presentation Graphs

In [ ]:
# A) Recording duration comparison
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x='label', y='duration_sec', palette='Set2')
sns.stripplot(data=df, x='label', y='duration_sec', color='black', alpha=0.45)
plt.title('Recording Duration by Class')
plt.xlabel('Class')
plt.ylabel('Duration (seconds)')
plt.show()

# B) Mean profile for selected channels
selected_means = [f'{ch}_mean' for ch in ['Fp1','Fp2','F3','F4','C3','C4','P3','P4','O1','O2'] if f'{ch}_mean' in df.columns]
if len(selected_means) >= 2:
    profile = df.groupby('label')[selected_means].mean().T
    profile.columns = [c.replace('_mean','') for c in profile.columns]
    plt.figure(figsize=(12,6))
    for cls in profile.columns:
        plt.plot(profile.index.str.replace('_mean',''), profile[cls], marker='o', label=cls)
    plt.xticks(rotation=45)
    plt.title('Channel-wise Mean Signal Profile by Class')
    plt.xlabel('Channel')
    plt.ylabel('Average Signal Mean')
    plt.legend()
    plt.tight_layout()
    plt.show()

# C) PCA 2D class separation
feature_cols = [c for c in df.columns if c not in ['filename', 'label']]
X = df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
y = df['label']

if X.shape[1] >= 2 and X.shape[0] >= 3:
    X_scaled = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    pca_df = pd.DataFrame({'PC1': X_pca[:,0], 'PC2': X_pca[:,1], 'label': y})

    plt.figure(figsize=(9,6))
    sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='label', s=120, alpha=0.85)
    plt.title(f'PCA Projection of EEG Features (Explained Var: {pca.explained_variance_ratio_.sum():.2%})')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend(title='Class')
    plt.show()

In [ ]:
print('Notebook ready. You can now Run All cells for complete analysis graphs.')